# MarsWalk Jezero — Multi-Scale Pruning Funnel Sweet-Spot Explorer (Colab)

Explores odd-sol tuning MAE (`mae_tune`) across the 4 funnel hyperparameters `(p, tau_max, h_min, eps)` on the `2 x 2 km` Western Delta Front USGS HiRISE `1.0 m/px` DTM window (`SRC-USGS-HIRISE-DTM-1M`).

- **Tuning split**: Perseverance waypoints with `sol % 2 == 1` (`odd_sols_only_tune`).
- **Held-out test split**: `sol % 2 == 0` (`even_sols_only`), evaluated only AFTER parameter lock.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

REPO_ROOT = Path("..").resolve()
SWEEP_PATH = REPO_ROOT / "bench" / "results" / "b3_sweep.json"
if not SWEEP_PATH.exists():
    from marswalk.bench.e2b_sweep import run_sweet_spot_sweep_128

    sweep_data = run_sweet_spot_sweep_128(REPO_ROOT)
else:
    sweep_data = json.loads(SWEEP_PATH.read_text(encoding="utf-8"))

print("Sweet Spot:", json.dumps(sweep_data["sweet_spot"], indent=2))

In [ ]:
records = [
    r
    for r in sweep_data["records"]
    if r["b3_persistent_hazard_pass"] and r["b9_entropy_floor_pass"]
]


def marginal_curve(param_name: str):
    vals = sorted({r["params"][param_name] for r in records})
    means = [
        sum(r["mae_tune"] for r in records if r["params"][param_name] == v)
        / sum(1 for r in records if r["params"][param_name] == v)
        for v in vals
    ]
    mins = [min(r["mae_tune"] for r in records if r["params"][param_name] == v) for v in vals]
    return vals, means, mins


fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, pname in zip(axes.flat, ["p", "tau_max", "h_min", "eps"], strict=True):
    xs, mean_ys, min_ys = marginal_curve(pname)
    ax.plot(xs, mean_ys, marker="o", label="Mean odd-sol MAE (deg)")
    ax.plot(xs, min_ys, marker="s", linestyle="--", label="Best odd-sol MAE (deg)")
    ax.set_title(f"Odd-Sol Tuning MAE vs {pname}")
    ax.set_xlabel(pname)
    ax.set_ylabel("MAE vs rover_tilt_deg (deg)")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from marswalk.bench.eval import evaluate_window_b5
from marswalk.engine.funnel import FunnelParams

sp = sweep_data["sweet_spot"]["params"]
locked_params = FunnelParams(p=sp["p"], tau_max=sp["tau_max"], h_min=sp["h_min"], eps=sp["eps"])
test_res = evaluate_window_b5(REPO_ROOT, split="test_even", params=locked_params)
print("Held-out EVEN test sols B5 result:", json.dumps(test_res, indent=2))